# MODUL PRAKTIKUM BIG DATA
## Pemrosesan Terdistribusi PySpark RDD di Databricks Community Edition

Memproses data besar (*Big Data*) menggunakan **PySpark RDD (Resilient Distributed Dataset)** di platform cloud **Databricks Community Edition**.

---

### 📌 Dataset yang Digunakan
* **Nama Dataset:** UCI / Kaggle – *Online Retail II Dataset*
* **Deskripsi:** Dataset transaksi nyata e-commerce ritel sebanyak lebih dari 500.000 baris record transaksi.
* **Tautan Dataset:**
  * **Kaggle:** [UCI Online Retail II Dataset](https://www.kaggle.com/datasets/vijayuv/onlineretail)
  * **UCI Direct Link:** [UCI Machine Learning Repository Zip/CSV](https://archive.ics.uci.edu/ml/machine-learning-databases/00352/)

## 1. Pengambilan Dataset Langsung ke Databricks DBFS

### Opsi A: Mengunduh Langsung via URL Publik (Wget)
Gunakan opsi ini untuk mengunduh dataset secara langsung tanpa perlu login.

In [1]:
# 1. Unduh dataset langsung ke environment Google Colab
!wget https://archive.ics.uci.edu/ml/machine-learning-databases/00502/online_retail_II.xlsx -O /content/online_retail.xlsx

# 2. Verifikasi keberadaan file di Colab
!ls -lh /content/online_retail.xlsx

--2026-11-08 04:35:25--  https://archive.ics.uci.edu/ml/machine-learning-databases/00502/online_retail_II.xlsx
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘/content/online_retail.xlsx’

/content/online_ret     [   <=>              ]  43.51M  77.4MB/s    in 0.6s    

2026-11-08 04:35:26 (77.4 MB/s) - ‘/content/online_retail.xlsx’ saved [45622278]

-rw-r--r-- 1 root root 44M Nov  8 04:35 /content/online_retail.xlsx


## 2. Inisialisasi & Verifikasi SparkContext

In [2]:
from pyspark.sql import SparkSession

# 1. Inisialisasi SparkSession secara lokal
spark = SparkSession.builder \
    .appName("Praktikum5_OnlineRetail_Luthfi") \
    .master("local[*]") \
    .getOrCreate()

# 2. Mendapatkan SparkContext dari SparkSession
sc = spark.sparkContext

# 3. Verifikasi (Menyesuaikan dengan instruksi soal Anda)
print("--- INFORMASI LOCAL SPARK ---")
print(f"Versi PySpark  : {spark.version}")
print(f"Master Node    : {sc.master}")

--- INFORMASI LOCAL SPARK ---
Versi PySpark  : 4.0.5
Master Node    : local[*]


## 3. Membaca Dataset dari DBFS ke PySpark RDD

In [6]:
import pandas as pd

# =========================================================================
# 3. Membaca Dataset dari DBFS ke PySpark RDD (Colab Environment)
# =========================================================================

# Proses 1: Konversi file Excel ke CSV untuk dibaca oleh PySpark
print("Proses 1: Mengonversi file Excel ke CSV... (Tunggu sekitar 1-2 menit)")
df_excel = pd.read_excel('/content/online_retail.xlsx')
df_excel.to_csv('/content/online_retail.csv', index=False)
print("Proses 1 Selesai! File online_retail.csv berhasil dibuat.\n")

# Proses 2: Membaca dataset CSV ke PySpark DataFrame
print("Proses 2: Membaca dataset CSV ke PySpark DataFrame...")
file_path = "/content/online_retail.csv"

df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

# Proses 3: Pembersihan data awal (Hapus baris dengan nilai NULL pada kolom vital)
df_clean = df_raw.dropna(subset=["Invoice", "Customer ID", "Price", "Quantity"])

# Proses 4: Konversi DataFrame ke RDD Tuple
# Format Tuple: (0:Invoice, 1:StockCode, 2:Description, 3:Quantity, 4:Price, 5:CustomerID, 6:Country, 7:InvoiceDate)
print("Proses 3: Mengonversi DataFrame ke RDD Tuple dan Partisi...")
rdd_raw = df_clean.rdd.map(lambda row: (
    str(row['Invoice']),
    str(row['StockCode']),
    str(row['Description']),
    int(row['Quantity']),
    float(row['Price']),
    str(row['Customer ID']),
    str(row['Country']),
    str(row['InvoiceDate'])
))

# Proses 5: Optimasi Partisi Data terdistribusi
rdd = rdd_raw.repartition(8)

# Proses 6: Verifikasi Hasil
print(f"Jumlah Total Record Terbersih: {rdd.count():,}")
print("\nContoh 3 Transaksi Pertama:")
for row in rdd.take(3):
    print(row)

Proses 1: Mengonversi file Excel ke CSV... (Tunggu sekitar 1-2 menit)
Proses 1 Selesai! File online_retail.csv berhasil dibuat.

Proses 2: Membaca dataset CSV ke PySpark DataFrame...
Proses 3: Mengonversi DataFrame ke RDD Tuple dan Partisi...
Jumlah Total Record Terbersih: 417,534

Contoh 3 Transaksi Pertama:
('489438', '21411', 'GINGHAM HEART  DOORSTOP RED', 32, 2.5, '18102.0', 'United Kingdom', '2009-12-01 09:24:00')
('489438', '84031A', 'CHARLIE+LOLA RED HOT WATER BOTTLE ', 56, 3.0, '18102.0', 'United Kingdom', '2009-12-01 09:24:00')
('489438', '84031B', 'CHARLIE LOLA BLUE HOT WATER BOTTLE ', 56, 3.0, '18102.0', 'United Kingdom', '2009-12-01 09:24:00')


## 4. Transformasi RDD (`filter` & `map`)

In [7]:
# 4. Transformasi RDD (filter & map)

# Filter transaksi valid dari UK (Qty & Price > 0)
rdd_uk_valid = rdd.filter(lambda x: x[3] > 0 and x[4] > 0 and x[6] == 'United Kingdom')

# Buat Pair RDD (Invoice, TotalSpent)
rdd_invoice_spent = rdd_uk_valid.map(lambda x: (x[0], round(x[3] * x[4], 2)))

print(f"Jumlah Transaksi Valid UK: {rdd_uk_valid.count():,}")
print("Contoh 3 Pair RDD:", rdd_invoice_spent.take(3))

Jumlah Transaksi Valid UK: 370,929
Contoh 3 Pair RDD: [('489438', 80.0), ('489438', 168.0), ('489438', 168.0)]


## 5. Agregasi RDD (`reduceByKey` & Sorting)

In [8]:
import time

# Hitung total belanja per Invoice ID
start_time = time.time()
rdd_total_per_invoice = rdd_invoice_spent.reduceByKey(lambda a, b: a + b)

# Urutkan transaksi dari nilai tertinggi
rdd_sorted_invoice = rdd_total_per_invoice.sortBy(lambda x: x[1], ascending=False)

top_10_invoices = rdd_sorted_invoice.take(10)
execution_time = time.time() - start_time

print(f"Waktu Eksekusi Agregasi RDD: {execution_time:.4f} detik\n")
print("--- TOP 10 INVOICE DENGAN TRANSAKSI TERBESAR ---")
for invoice, total in top_10_invoices:
    print(f"Invoice ID: {invoice:<10} | Total Belanja: £ {total:,.2f}")

Waktu Eksekusi Agregasi RDD: 5.0038 detik

--- TOP 10 INVOICE DENGAN TRANSAKSI TERBESAR ---
Invoice ID: 524181     | Total Belanja: £ 33,167.80
Invoice ID: 526934     | Total Belanja: £ 26,007.08
Invoice ID: 515944     | Total Belanja: £ 22,863.36
Invoice ID: 517731     | Total Belanja: £ 21,984.00
Invoice ID: 494243     | Total Belanja: £ 18,532.30
Invoice ID: 531866     | Total Belanja: £ 18,122.17
Invoice ID: 530715     | Total Belanja: £ 17,022.24
Invoice ID: 537659     | Total Belanja: £ 15,885.49
Invoice ID: 535104     | Total Belanja: £ 15,070.22
Invoice ID: 527870     | Total Belanja: £ 14,583.78


## 6. Optimasi RDD: Caching & Persistence

In [9]:
from pyspark import StorageLevel
import time

# 6. Optimasi RDD: Caching & Persistence

# Simpan RDD ke RAM
rdd_sorted_invoice.persist(StorageLevel.MEMORY_ONLY)

# Eksekusi Aksi Pertama (Cache Warmup)
t0 = time.time()
count_1 = rdd_sorted_invoice.count()
t1 = time.time() - t0

# Eksekusi Aksi Kedua (Membaca langsung dari RAM)
t2 = time.time()
count_2 = rdd_sorted_invoice.count()
t3 = time.time() - t2

print(f"Eksekusi Aksi Pertama (Tanpa Cache Warmup) : {t1:.4f} detik")
print(f"Eksekusi Aksi Kedua (Menggunakan Cache)    : {t3:.4f} detik")

# Lepaskan memori setelah selesai
rdd_sorted_invoice.unpersist()

Eksekusi Aksi Pertama (Tanpa Cache Warmup) : 0.7174 detik
Eksekusi Aksi Kedua (Menggunakan Cache)    : 0.3896 detik


PythonRDD[69] at RDD at PythonRDD.scala:57

## 7. Analisis Produk Paling Laris

In [10]:
# 7. Analisis Produk Paling Laris

# Analisis Top 5 Produk Paling Banyak Terjual (Total Kuantitas)
# Pair RDD: (Description, Quantity)
rdd_product_qty = rdd_uk_valid.map(lambda x: (x[2], x[3])) \
                              .reduceByKey(lambda a, b: a + b) \
                              .sortBy(lambda x: x[1], ascending=False)

print("--- TOP 5 PRODUK TERLARIS ---")
for product, total_qty in rdd_product_qty.take(5):
    print(f"Produk: {product:<40} | Jumlah Terjual: {total_qty:,} unit")

--- TOP 5 PRODUK TERLARIS ---
Produk: WHITE HANGING HEART T-LIGHT HOLDER       | Jumlah Terjual: 51,679 unit
Produk: WORLD WAR 2 GLIDERS ASSTD DESIGNS        | Jumlah Terjual: 51,538 unit
Produk: BROCADE RING PURSE                       | Jumlah Terjual: 48,014 unit
Produk: ASSORTED COLOUR BIRD ORNAMENT            | Jumlah Terjual: 41,529 unit
Produk: PACK OF 72 RETRO SPOT CAKE CASES         | Jumlah Terjual: 35,347 unit


-----------------------------------------------------------------------------------

## 8. Tugas Mandiri Mahasiswa

-----------------------------------------------------------------------------------

#### Dilarang Menyimpan data ataupun memproses data di komputer lokal.
### Gunakan Google Colab, Databricks, atau Kaggle, atau cloud service yang lain.

# -----------------------------------------------------------------------------------
## 8. TUGAS MANDIRI MAHASISWA (PYSPARK RDD BENCHMARK & ANALYTICS)
# -----------------------------------------------------------------------------------
### Petunjuk Umum:
### 1. Kerjakan tugas-tugas di bawah ini menggunakan transformasi dan aksi PySpark RDD.
### 2. Jangan mengubah data sumber utama (rdd / rdd_uk_valid).
### 3. Tuliskan skrip dan jalankan sel untuk menampilkan output jawaban Anda!
# -----------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
## KATEGORI A: ANALISIS DATA & EKSPLORASI BISNIS (BUSINESS INTELLIGENCE)
# ------------------------------------------------------------------------------

### TUGAS A1: Analisis Pelanggan Utama Luar Negeri (Non-UK)
#### Instruksi:
#### 1. Filter dataset untuk mengambil transaksi yang berasal dari LUAR negara 'United Kingdom' (Country != 'United Kingdom').
#### 2. Buat Pair RDD dengan Key = CustomerID dan Value = Total Spent (Quantity * UnitPrice).
#### 3. Hitung total belanja setiap CustomerID dan tampilkan Top 5 Pelanggan dengan pengeluaran terbesar!

In [11]:
# TUGAS A1: Analisis Pelanggan Utama Luar Negeri (Non-UK)
# Instruksi:
# 1. Filter dataset untuk mengambil transaksi yang berasal dari LUAR negara 'United Kingdom' (Country != 'United Kingdom').
# 2. Buat Pair RDD dengan Key = CustomerID dan Value = Total Spent (Quantity * UnitPrice).
# 3. Hitung total belanja setiap CustomerID dan tampilkan Top 5 Pelanggan dengan pengeluaran terbesar!
#
# Tulis kode Anda di bawah ini:
# ================= TUGAS A1 =================

# 1. Filter Non-UK dan pastikan transaksi valid (Qty & Price > 0)
rdd_non_uk = rdd.filter(lambda x: x[6] != 'United Kingdom' and x[3] > 0 and x[4] > 0)

# 2. Buat Pair RDD (CustomerID, Total Spent)
rdd_cust_spent = rdd_non_uk.map(lambda x: (x[5], x[3] * x[4]))

# 3. Hitung total belanja per CustomerID, urutkan, dan ambil Top 5
top_5_foreign = rdd_cust_spent.reduceByKey(lambda a, b: a + b) \
                              .sortBy(lambda x: x[1], ascending=False) \
                              .take(5)

print("--- TOP 5 PELANGGAN LUAR NEGERI (NON-UK) ---")
for cust, spent in top_5_foreign:
    print(f"CustomerID: {cust:<10} | Total Pengeluaran: £ {spent:,.2f}")

--- TOP 5 PELANGGAN LUAR NEGERI (NON-UK) ---
CustomerID: 14646.0    | Total Pengeluaran: £ 248,396.50
CustomerID: 14156.0    | Total Pengeluaran: £ 196,566.74
CustomerID: 14911.0    | Total Pengeluaran: £ 152,147.57
CustomerID: 13902.0    | Total Pengeluaran: £ 34,095.26
CustomerID: 12482.0    | Total Pengeluaran: £ 23,691.40


In [12]:
# TUGAS A2: Analisis Potensi Kerugian Transaksi Pembatalan/Retur
# Instruksi:
# 1. Filter dataset untuk mencari transaksi pembatalan, yaitu yang InvoiceNo diawali huruf 'C' ATAU Quantity < 0.
# 2. Hitung berapa jumlah total transaksi yang dibatalkan.
# 3. Hitung total estimasi nilai kerugian (Total Nilai Transaksi yang dibatalkan) dalam GBP (£).
#
# Tulis kode Anda di bawah ini:
# ================= TUGAS A2 =================

# 1. Filter transaksi pembatalan (Invoice diawali 'C' atau Quantity < 0)
rdd_canceled = rdd.filter(lambda x: str(x[0]).upper().startswith('C') or x[3] < 0)

# 2. Hitung jumlah total transaksi yang dibatalkan
total_cancel_count = rdd_canceled.count()

# 3. Hitung total estimasi kerugian (Absolut dari Qty * Price)
total_loss = rdd_canceled.map(lambda x: abs(x[3]) * x[4]).sum()

print("--- TUGAS A2: ANALISIS POTENSI KERUGIAN ---")
print(f"Jumlah Transaksi Dibatalkan : {total_cancel_count:,} transaksi")
print(f"Total Estimasi Kerugian     : £ {total_loss:,.2f}")

--- TUGAS A2: ANALISIS POTENSI KERUGIAN ---
Jumlah Transaksi Dibatalkan : 9,839 transaksi
Total Estimasi Kerugian     : £ 483,794.70



# ------------------------------------------------------------------------------
### KATEGORI B: OPTIMASI PERFORMA & BENCHMARK RDD (PERFORMANCE ENGINEERING)
# ------------------------------------------------------------------------------

In [13]:
# TUGAS B1: Eksperimen Pengaruh Partisi Terhadap Waktu Eksekusi
# Instruksi:
# 1. Gunakan fungsi .repartition() untuk mengubah RDD ke dalam 3 variasi jumlah partisi: 2, 8, dan 16.
# 2. Jalankan operasi agregasi reduceByKey() pada masing-masing variasi partisi tersebut.
# 3. Ukur dan cetak waktu eksekusi (runtime) masing-masing variasi menggunakan pustaka `time`.
#
# Tulis kode Anda di bawah ini:
import time

# ================= TUGAS B1 =================
print("--- TUGAS B1: BENCHMARK PARTISI ---")

# Variasi jumlah partisi yang akan diuji
variasi_partisi = [2, 8, 16]

for p in variasi_partisi:
    # 1. Ubah jumlah partisi RDD
    rdd_partitioned = rdd_invoice_spent.repartition(p)

    # Mulai ukur waktu
    start_time = time.time()

    # 2. Jalankan reduceByKey, dan gunakan count() untuk memicu eksekusi (Lazy Evaluation)
    rdd_partitioned.reduceByKey(lambda a, b: a + b).count()

    # Selesai ukur waktu
    waktu_eksekusi = time.time() - start_time

    # 3. Cetak hasil
    print(f"Waktu Eksekusi ({p:2} Partisi) : {waktu_eksekusi:.4f} detik")

--- TUGAS B1: BENCHMARK PARTISI ---
Waktu Eksekusi ( 2 Partisi) : 3.8120 detik
Waktu Eksekusi ( 8 Partisi) : 3.1740 detik
Waktu Eksekusi (16 Partisi) : 3.4637 detik


In [14]:
# TUGAS B2: Perbandingan Performa reduceByKey vs groupByKey
# Instruksi:
# 1. Buat Pair RDD dengan Key = Description (Nama Produk) dan Value = Quantity.
# 2. Hitung total kuantitas terjual per produk menggunakan `reduceByKey()`, catat waktu eksekusinya.
# 3. Hitung total kuantitas terjual per produk menggunakan `groupByKey()` + `mapValues(sum)`, catat waktu eksekusinya.
# 4. Tampilkan perbandingan selisih waktu eksekusi kedua fungsi tersebut.
#
# Tulis kode Anda di bawah ini:
import time

# ================= TUGAS B2 =================
print("--- TUGAS B2: reduceByKey vs groupByKey ---")

# 1. Buat Pair RDD (Description, Quantity) menggunakan data transaksi valid
rdd_desc_qty = rdd_uk_valid.map(lambda x: (x[2], x[3]))

# 2. Hitung total menggunakan reduceByKey()
start_rbk = time.time()
rdd_desc_qty.reduceByKey(lambda a, b: a + b).count()
time_rbk = time.time() - start_rbk
print(f"Waktu eksekusi reduceByKey() : {time_rbk:.4f} detik")

# 3. Hitung total menggunakan groupByKey() + mapValues(sum)
start_gbk = time.time()
rdd_desc_qty.groupByKey().mapValues(sum).count()
time_gbk = time.time() - start_gbk
print(f"Waktu eksekusi groupByKey()  : {time_gbk:.4f} detik")

# 4. Tampilkan perbandingan selisih waktu
selisih = abs(time_gbk - time_rbk)
print(f"Selisih waktu eksekusi       : {selisih:.4f} detik")

if time_rbk < time_gbk:
    print("\nAnalisis: reduceByKey() lebih cepat karena memproses data di tiap partisi lokal (map-side combiner) sebelum melakukan shuffle di jaringan.")
else:
    print("\nAnalisis: Fluktuasi lokal dapat terjadi, namun secara teori Big Data, reduceByKey() jauh lebih optimal dan aman dari risiko Out of Memory dibanding groupByKey().")

--- TUGAS B2: reduceByKey vs groupByKey ---
Waktu eksekusi reduceByKey() : 1.7810 detik
Waktu eksekusi groupByKey()  : 1.5339 detik
Selisih waktu eksekusi       : 0.2471 detik

Analisis: Fluktuasi lokal dapat terjadi, namun secara teori Big Data, reduceByKey() jauh lebih optimal dan aman dari risiko Out of Memory dibanding groupByKey().


# ------------------------------------------------------------------------------
### KATEGORI C: PEMROSESAN STRING & WAKTU (TEXT & DATE MANIPULATION)
# ------------------------------------------------------------------------------

In [15]:
# TUGAS C1: Analisis Jam Sibuk Transaksi (Peak Transaction Hours)
# Instruksi:
# 1. Buat fungsi parser atau ekstraksi Lambda untuk mengambil bagian 'Jam' (Hour) dari kolom `InvoiceDate` (Format: "MM/DD/YYYY HH:MM").
# 2. Bentuk Pair RDD (Jam, 1) dan gunakan reduceByKey untuk menghitung total frekuensi transaksi per jam.
# 3. Urutkan hasil dari jam yang paling sibuk dan tampilkan Top 3 Jam Sibuk!
#
# Tulis kode Anda di bawah ini:
# ================= TUGAS C1 =================

# 1 & 2. Ekstrak Jam (Hour) dari InvoiceDate (x[7]) dan bentuk Pair RDD (Jam, 1)
# Logika: split(' ') memisahkan tanggal & waktu, lalu split(':')[0] mengambil angka jam saja
rdd_hour = rdd.map(lambda x: (str(x[7]).split(' ')[1].split(':')[0], 1))

# 3. Hitung total frekuensi transaksi per jam, urutkan menurun, ambil Top 3
top_3_hours = rdd_hour.reduceByKey(lambda a, b: a + b) \
                      .sortBy(lambda x: x[1], ascending=False) \
                      .take(3)

print("--- TUGAS C1: TOP 3 JAM SIBUK TRANSAKSI ---")
for hour, count in top_3_hours:
    print(f"Pukul {hour}:00 | Jumlah Transaksi: {count:,}")

--- TUGAS C1: TOP 3 JAM SIBUK TRANSAKSI ---
Pukul 12:00 | Jumlah Transaksi: 73,677
Pukul 13:00 | Jumlah Transaksi: 68,900
Pukul 14:00 | Jumlah Transaksi: 57,281


In [16]:
# TUGAS C2: Pengelompokan & Rata-rata Harga Produk Spesifik
# Instruksi:
# 1. Filter produk pada `rdd_uk_valid` yang deskripsi produknya mengandung kata kunci "HEART" (case-insensitive).
# 2. Hitung Rata-Rata Harga Jual (`UnitPrice`) untuk setiap varian produk yang mengandung kata "HEART".
# 3. Tampilkan total varian produk yang ditemukan beserta contoh 5 produk dan rata-rata harganya!
#
# Tulis kode Anda di bawah ini:
# ================= TUGAS C2 =================

# 1. Filter deskripsi produk yang mengandung kata "HEART" (case-insensitive)
rdd_heart = rdd_uk_valid.filter(lambda x: x[2] is not None and 'HEART' in str(x[2]).upper())

# 2. Hitung Rata-Rata Harga Jual (UnitPrice / Price)
# Bentuk tuple awal: (Description, (Price, 1))
rdd_heart_pair = rdd_heart.map(lambda x: (x[2], (x[4], 1)))

# Akumulasi total harga dan total kemunculan per varian produk
rdd_heart_sum = rdd_heart_pair.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# Hitung rata-rata: (Total Harga / Total Kemunculan)
rdd_heart_avg = rdd_heart_sum.map(lambda x: (x[0], round(x[1][0] / x[1][1], 2)))

# 3. Hitung total varian dan ambil 5 contoh
total_variants = rdd_heart_avg.count()
sample_products = rdd_heart_avg.take(5)

print("--- TUGAS C2: RATA-RATA HARGA PRODUK 'HEART' ---")
print(f"Total Varian Produk Ditemukan: {total_variants} varian\n")
print("Contoh 5 Produk:")
for product, avg_price in sample_products:
    print(f"- {product:<40} | Rata-rata Harga: £ {avg_price:.2f}")

--- TUGAS C2: RATA-RATA HARGA PRODUK 'HEART' ---
Total Varian Produk Ditemukan: 265 varian

Contoh 5 Produk:
- CREAM SWEETHEART SHELF + HOOKS           | Rata-rata Harga: £ 7.71
- CREAM SWEETHEART MINI CHEST              | Rata-rata Harga: £ 12.67
- LOVE HEART POCKET WARMER                 | Rata-rata Harga: £ 1.55
- LAVENDER SCENTED FABRIC HEART            | Rata-rata Harga: £ 1.25
- TWO EMBOSSED HEART MINI DRAWERS          | Rata-rata Harga: £ 3.75


# ------------------------------------------------------------------------------
### KATEGORI D: OPERASI MULTISET & SET OPERATIONS (ADVANCED RDD CONCEPTS)
# ------------------------------------------------------------------------------

In [17]:
# TUGAS D1: Eksklusivitas Pasar (Operasi Himpunan RDD `subtract`)
# Instruksi:
# 1. Dapatkan daftar Nama Produk unik (`distinct()`) yang pernah dibeli di negara 'Germany'.
# 2. Dapatkan daftar Nama Produk unik (`distinct()`) yang pernah dibeli di negara 'France'.
# 3. Gunakan fungsi himpunan RDD `.subtract()` untuk menemukan produk yang dibeli di 'Germany' TETAPI TIDAK PERNAH dibeli di 'France'.
# 4. Tampilkan jumlah produk eksklusif tersebut dan cetak 5 contoh produknya!
#
# Tulis kode Anda di bawah ini:
# ================= TUGAS D1 =================

# 1. Produk unik yang pernah dibeli di Germany
rdd_germany = rdd.filter(lambda x: x[6] == 'Germany' and x[3] > 0) \
                 .map(lambda x: x[2]).distinct()

# 2. Produk unik yang pernah dibeli di France
rdd_france = rdd.filter(lambda x: x[6] == 'France' and x[3] > 0) \
                .map(lambda x: x[2]).distinct()

# 3. Gunakan subtract() untuk mencari produk di Germany tapi TIDAK di France
rdd_germany_only = rdd_germany.subtract(rdd_france)

# 4. Hitung jumlah dan tampilkan 5 contohnya
total_exclusive = rdd_germany_only.count()
sample_exclusive = rdd_germany_only.take(5)

print("--- TUGAS D1: PRODUK EKSKLUSIF GERMANY (VS FRANCE) ---")
print(f"Jumlah Produk Eksklusif: {total_exclusive} produk\n")
print("Contoh 5 Produk:")
for product in sample_exclusive:
    print(f"- {product}")

--- TUGAS D1: PRODUK EKSKLUSIF GERMANY (VS FRANCE) ---
Jumlah Produk Eksklusif: 621 produk

Contoh 5 Produk:
- BABY BOOM RIBBONS 
- ASSORTED TUTTI FRUTTI  FOB NOTEBOOK
- JUMBO BAG SCANDINAVIAN PAISLEY
- DOORMAT BLACK FLOCK 
- HAND WARMER OWL DESIGN
